In [17]:
#4. Smart-Grid AI — Power Station Stability Prediction

#Context: The National Energy Grid|Sensor records: 100
#Inputs: Temperature (C); Wind_Speed (km/h); Current_Load (MW)
#Target used for the runnable regression example: Grid_Stability_Score — predicted grid stability score (0–100).

# ANN Level 1 - FULL PROGRAM (Worksheet + Answers combined)
# 4. Smart-Grid AI — Power Station Stability Prediction (Regression)

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
torch.manual_seed(42)
np.random.seed(42)
print("Libraries imported successfully! PyTorch Version:", torch.__version__)

Libraries imported successfully! PyTorch Version: 2.13.0+cpu


In [18]:
# [CELL 2] Data Pipeline, Normalization & Tensor Conversion

if not os.path.exists('smart_grid_stations.csv'):
    n = 100
    demo_df = pd.DataFrame({
        'Temperature': np.random.uniform(5, 45, n),
        'Wind_Speed': np.random.uniform(0, 60, n),
        'Current_Load': np.random.uniform(50, 900, n),
    })
    # --- ANSWER: Create the regression target from the sensor inputs --
    demo_df['Grid_Stability_Score'] = np.clip(
        90 - 0.45*demo_df['Temperature'] + 0.30*demo_df['Wind_Speed']
        - 0.06*demo_df['Current_Load'] + np.random.normal(0, 4, n), 0, 100
    )
    for col in ['Temperature', 'Wind_Speed']:
        mask = np.random.rand(n) < 0.03
        demo_df.loc[mask, col] = np.nan
    demo_df.to_csv('smart_grid_stations.csv', index=False)

df = pd.read_csv('smart_grid_stations.csv')

feature_cols = ['Temperature', 'Wind_Speed', 'Current_Load']
X = df[feature_cols].values
y = df['Grid_Stability_Score'].values

imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
print(f"X_train shape: {X_train_tensor.shape}, y_train shape: {y_train_tensor.shape}")

X_train shape: torch.Size([80, 3]), y_train shape: torch.Size([80, 1])


In [19]:
# [CELL 3] Designing the ANN Architecture for Regression

class GridStabilityRegressor(nn.Module):
    def __init__(self):
        super(GridStabilityRegressor, self).__init__() 
        self.linear1 = nn.Linear(3, 32)
        self.linear2 = nn.Linear(32, 16)
        self.linear3 = nn.Linear(16, 1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        return x
model = GridStabilityRegressor()
print(model)


GridStabilityRegressor(
  (linear1): Linear(in_features=3, out_features=32, bias=True)
  (linear2): Linear(in_features=32, out_features=16, bias=True)
  (linear3): Linear(in_features=16, out_features=1, bias=True)
  (relu): ReLU()
)


In [12]:
# [CELL 4] Loss Function & Optimizer

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print("Loss function: MSELoss | Optimizer: Adam")

Loss function: MSELoss | Optimizer: Adam


In [20]:
# [CELL 5] The Training Loop

num_epochs = 100
batch_size = 32
print("Training started...\n")
for epoch in range(num_epochs):
    for i in range(0, len(X_train_tensor), batch_size):
        X_batch = X_train_tensor[i:i+batch_size]
        y_batch = y_train_tensor[i:i+batch_size]
        predictions = model(X_batch)
        loss = loss_fn(predictions, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], MSE Loss: {loss.item():.4f}")
print("\nTraining complete!")

Training started...

Epoch [20/100], MSE Loss: 3944.8408
Epoch [40/100], MSE Loss: 3944.8408
Epoch [60/100], MSE Loss: 3944.8408
Epoch [80/100], MSE Loss: 3944.8408
Epoch [100/100], MSE Loss: 3944.8408

Training complete!


In [21]:
# [CELL 6] The Evaluation Loop

model.eval()
with torch.no_grad():
    test_predictions = model(X_test_tensor)
test_mse = loss_fn(test_predictions, y_test_tensor).item()
test_rmse = np.sqrt(test_mse)
print("Test Set Evaluation:")
print(f" - Mean Squared Error (MSE) : {test_mse:.4f}")
print(f" - Root Mean Squared Error (RMSE): {test_rmse:.4f}\n")
print(f"{'Actual':<15} {'Predicted':<15} {'Abs Error':<15}")
print("-" * 45)
with torch.no_grad():
    for i in range(5):
        actual = y_test_tensor[i].item()
        predicted = test_predictions[i].item()
        error = abs(actual - predicted)
        print(f"{actual:<15.2f} {predicted:<15.2f} {error:<15.2f}")

Test Set Evaluation:
 - Mean Squared Error (MSE) : 3354.5742
 - Root Mean Squared Error (RMSE): 57.9187

Actual          Predicted       Abs Error      
---------------------------------------------
94.69           0.31            94.38          
53.48           0.31            53.17          
44.76           0.25            44.51          
23.74           0.30            23.45          
85.23           0.35            84.88          


In [15]:
# [CELL 7] Real-World Inference

raw_input = np.array([[30.0, 18.0, 420.0]])
imputed_data = imputer.transform(raw_input)
scaled_data = scaler.transform(imputed_data)
input_tensor = torch.tensor(scaled_data, dtype=torch.float32)
model.eval()
with torch.no_grad():
    predicted_score = model(input_tensor).item()
print(f"Predicted Grid Stability Score: {predicted_score:.2f} / 100")


Predicted Grid Stability Score: 31.08 / 100
